# NYC Yellow Taxi — Trip Duration Model Development (January 2025)

**Goal:** find the most accurate model for the duration of a yellow-taxi trip, using only what is known
at pickup time: pickup zone, drop-off zone and pickup time.

**Evaluation protocol** (January 2025 only, chronological, fixed; nothing is retrained on more data):

| Split | Pickup dates | Used for |
|---|---|---|
| Train | Jan 1–20 | fitting every model and all learned statistics |
| Validation | Jan 21–25 | comparing models (baselines, RF baseline, Optuna trials) |
| Test | Jan 26–31 | **one** final evaluation of the selected model, next to the baselines |

Metrics: **RMSE** and **MAE** in minutes, tracked in MLflow.

**This notebook trains nothing.** It loads the data, checks the cleaning, explores it and states what
the exploration decides. Every model result is produced by the pipeline (`uv run nyc-taxi all`) and
**read from MLflow** here:

| MLflow run | Made by | Shown here |
|---|---|---|
| `baseline-overall-median`, `baseline-zone-pair-median`, `rf-baseline` | `nyc-taxi train` | baselines (validation) |
| `optuna-tuning` | `nyc-taxi tune` | Optuna trials: validation RMSE/MAE and model size |
| `select` | `nyc-taxi select` | the winner and the one test comparison |

The selection rule and its chart are shown in `model_serving_selection.ipynb`.

**Before running:** start MLflow (`docker compose up -d postgres mlflow`) and run
`uv run nyc-taxi all` once.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import holidays
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src import baselines as bl  # baseline run names and readers
from src import evaluation as ev  # shared metrics and segments
from src import tracking  # MLflow run readers
from src.experiment import EXPERIMENT, TRACKING_URI
from src.features import FEATURES, build_features, build_zone_xy
from src.selection import SELECT_RUN_NAME, TEST_FILE
from src.train import (BASELINE_PARAMS, COLUMNS, DATA_PATH, DATA_URL, clean_trips, download_january,
                       split_by_time)
from src.tuning import TUNING_RUN_NAME

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT)  # fails fast if the MLflow server is not running
print("MLflow tracking server:", TRACKING_URI)

## 1.1 Raw data check: why trips are filtered

Before cleaning, look at the raw January file (all records, no filter) to justify the rule in
`src.train.clean_trips`: **keep trips of 1-60 minutes**. `trip_distance` and the fare columns are loaded
here only for these checks; they are not model features (they are only known after the trip).

In [ ]:
MIN_DURATION_MINUTES, MAX_DURATION_MINUTES = 1, 60  # same rule as src.train.clean_trips
MAX_DISTANCE_MILES = 100  # a NYC yellow-taxi trip longer than this is not plausible
TARGET = "duration"
BLUE, ORANGE = "#1f77b4", "#ff7f0e"
IMAGES = ROOT / "images"
IMAGES.mkdir(exist_ok=True)

# Columns for these checks only. The training download keeps 4 columns, so if the local file
# does not have them they are read from the TLC URL instead (~1 min, not saved).
CHECK_COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
                 "fare_amount", "tip_amount", "total_amount"]
if not DATA_PATH.exists():
    download_january()  # one-time download, ~1 min
has_columns = set(CHECK_COLUMNS) <= set(pq.read_schema(DATA_PATH).names)
trips = pd.read_parquet(DATA_PATH if has_columns else DATA_URL, columns=CHECK_COLUMNS)

trips[TARGET] = (
    trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60
trips["speed_mph"] = trips["trip_distance"] / (trips[TARGET] / 60)  # implied average speed
print(f"Raw records: {len(trips):,}")

### Duration

In [ ]:
fig, ax = plt.subplots()
trips[TARGET].clip(-10, 100).plot.hist(
    bins=110, ax=ax, color=BLUE, edgecolor="none"
)
for bound in (MIN_DURATION_MINUTES, MAX_DURATION_MINUTES):
    ax.axvline(bound, color=ORANGE, linewidth=1.5)
ax.set(
    title="Trip duration, January 2025 (clipped at -10 and 100 min)",
    xlabel="minutes",
    ylabel="trips",
)
ax.yaxis.set_major_formatter("{x:,.0f}")
ax.annotate(
    "keep 1-60 min", xy=(31, ax.get_ylim()[1] * 0.85), color=ORANGE, ha="center"
)
fig.tight_layout()
fig.savefig(IMAGES / "duration_histogram.png")
plt.show()

In [ ]:
share_kept = (
    trips[TARGET]
    .between(MIN_DURATION_MINUTES, MAX_DURATION_MINUTES)
    .mean()
)
print(f"Trips between 1 and 60 minutes: {share_kept:.1%}")

What is in the removed tails? For each duration band: how many trips, the median distance, the share
with zero distance and the median implied speed (distance / duration).

In [ ]:
bands = pd.cut(
    trips[TARGET],
    [-np.inf, 0, 1, 60, 180, 24 * 60, np.inf],
    right=False,
    labels=["< 0 min", "0-1 min", "1-60 min (kept)", "1-3 h", "3-24 h", ">= 24 h"],
)
by_band = trips.groupby(bands, observed=True).agg(
    trips=(TARGET, "size"),
    median_miles=("trip_distance", "median"),
    zero_distance=("trip_distance", lambda s: (s == 0).mean()),
    median_mph=("speed_mph", "median"),
)
by_band.insert(1, "share", by_band["trips"] / len(trips))
by_band.style.format({"trips": "{:,}", "share": "{:.2%}", "median_miles": "{:.2f}",
                      "zero_distance": "{:.1%}", "median_mph": "{:,.1f}"})

- **Negative duration:** drop-off before pickup, a clock error. These cannot be a valid target.
- **Under 1 minute:** most have zero distance: cancelled rides, meter tests, or the meter started and
  stopped at once. They are not real trips.
- **3 hours and more:** the median speed is far below walking pace (well under 1 mph) for a
  short distance: the meter was left running. The recorded duration is not the travel time.
- **1-3 hours:** these are mostly real long trips (about 17 miles, ~15 mph: airports, traffic). They
  are about 0.9% of records. They are removed to keep the model on regular city trips, and because a few
  extreme targets inflate the squared error (RMSE) that the models minimise. The API is therefore scoped
  to trips of up to 60 minutes.

### Distance

In [ ]:
trips["trip_distance"].describe(
    percentiles=[0.01, 0.026, 0.5, 0.99, 0.9999]
).to_frame().T

In [ ]:
too_far = trips["trip_distance"] > MAX_DISTANCE_MILES
kept = trips[TARGET].between(MIN_DURATION_MINUTES, MAX_DURATION_MINUTES)
print(f"Trips over {MAX_DISTANCE_MILES} miles:         {too_far.sum():>8,}")
print(f"  median duration:              {trips.loc[too_far, TARGET].median():>8.1f} min")
print(f"  median implied speed:         {trips.loc[too_far, 'speed_mph'].median():>8,.0f} mph")
print(f"  still inside 1-60 min:        {(too_far & kept).sum():>8,}")
print(f"Zero-distance trips:            {(trips['trip_distance'] == 0).mean():>8.1%}")

- About 2.6% of records have **zero distance**, and 99.99% of trips are under ~60 miles. The maximum is
hundreds of thousands of miles.
- Trips over 100 miles last about 20 minutes (median), which means an implied speed of more than
100,000 mph. The recorded distance is wrong (an odometer or meter error), not the trip.
- **The cleaning rule does not filter on distance**, and it does not need to: `trip_distance` is not a
feature and not the target. The model uses the zones only (`straight_dist` comes from zone centers), so
a wrong odometer value cannot reach the model. The ~120 trips over 100 miles that last 1-60 minutes are
kept: their zones and times are valid. The broken distances are one more reason not to use this column.

### Columns known only after the trip

The raw file also has the distance and the fare, tip and total amount. They follow the duration closely
(cell below, clean 1-60 minute trips), but they are only recorded when the trip ends, so a model that
predicts at pickup cannot use them.

In [ ]:
after_trip = ["trip_distance", "fare_amount", "tip_amount", "total_amount"]
kept_trips = trips.loc[kept]
leak_corr = kept_trips[after_trip].corrwith(kept_trips[TARGET], method="spearman")
print("Spearman correlation with duration (1-60 min trips):")
print(leak_corr.round(2).to_string())

In [ ]:
del trips, bands, too_far, kept, kept_trips  # free memory

## 2. Load and clean the data

The raw file is downloaded once into `data/` (only the 4 columns used). Cleaning (`src.train.clean_trips`):
keep pickups inside January 2025, compute `duration` in minutes, keep trips of 1–60 minutes, sort by
pickup time. These are fixed rules, not statistics learned from the data.

In [ ]:
if not DATA_PATH.exists():
    download_january()  # one-time download, ~1 min

raw = pd.read_parquet(DATA_PATH, columns=COLUMNS)
jan = clean_trips(raw)

pickup = raw["tpep_pickup_datetime"]
in_january = (pickup >= pd.Timestamp("2025-01-01")) & (pickup < pd.Timestamp("2025-02-01"))
print(f"Raw records:               {len(raw):>10,}")
print(f"Pickup outside January:    {(~in_january).sum():>10,}")
print(f"Duration outside 1-60 min: {in_january.sum() - len(jan):>10,}")
print(f"Clean trips kept:          {len(jan):>10,}  ({len(jan) / len(raw):.1%})")

## 3. Chronological split

In [ ]:
tr, va, te = split_by_time(jan)
split = pd.Series(np.select([tr, va], ["train", "validation"], "test"), index=jan.index)

pd.DataFrame({name: {"pickup dates": ev.describe_period(jan[mask]), "share": f"{mask.mean():.1%}"}
              for name, mask in [("train", tr), ("validation", va), ("test", te)]}).T

## 4. Exploration

The duration histogram uses all raw records (to justify the cleaning rule). Patterns used to design
features (hour of day) are looked at on the **training days only**.

In [ ]:
raw_minutes = (raw["tpep_dropoff_datetime"] - raw["tpep_pickup_datetime"]).dt.total_seconds() / 60
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].hist(raw_minutes.clip(-5, 120), bins=125, color="grey")
axes[0].axvspan(1, 60, color="tab:green", alpha=0.15, label="kept: 1-60 min")
axes[0].set(title="Raw trip duration (clipped to -5..120 min)", xlabel="minutes", ylabel="trips (log)",
            yscale="log")
axes[0].legend()

daily = jan.groupby([jan["tpep_pickup_datetime"].dt.day, split]).size().unstack(fill_value=0)
for name, color in [("train", "tab:blue"), ("validation", "tab:orange"), ("test", "tab:red")]:
    axes[1].bar(daily.index, daily[name], color=color, label=name)
axes[1].set(title="Clean trips per day", xlabel="day of January", ylabel="trips")
axes[1].legend()

train_hours = jan.loc[tr, "tpep_pickup_datetime"].dt.hour
jan.loc[tr].groupby(train_hours)["duration"].median().plot(ax=axes[2], marker="o")
axes[2].set(title="Median duration by pickup hour (train days)", xlabel="hour", ylabel="minutes")
plt.tight_layout()
plt.show()

del raw, raw_minutes, pickup, in_january  # free memory

### Time patterns (training days only)

Numbers behind the time features and the time-based split.

In [ ]:
train_trips = jan.loc[tr]
pickup_tr = train_trips["tpep_pickup_datetime"]

by_hour = train_trips.groupby(pickup_tr.dt.hour)["duration"].median()
print(f"Median duration by pickup hour: {by_hour.min():.1f} min at {by_hour.idxmin()}h, "
      f"{by_hour.max():.1f} min at {by_hour.idxmax()}h")

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = train_trips.groupby(pickup_tr.dt.day_name())["duration"].median().reindex(weekday_order)
print("Median duration by weekday (min):", by_weekday.round(1).to_dict())

holiday_dates = list(holidays.US(years=[2025]).keys())  # same calendar as src/features.py
is_holiday = pickup_tr.dt.date.isin(holiday_dates)
is_weekday = pickup_tr.dt.day_of_week < 5
morning = pickup_tr.dt.hour.isin([8, 9])  # 8-10 am
print("Holidays in the training days:", sorted({str(d) for d in pickup_tr[is_holiday].dt.date}))
for label, mask in [("all day", is_weekday), ("8-10 am", is_weekday & morning)]:
    print(f"Median duration {label}: weekday holidays "
          f"{train_trips.loc[mask & is_holiday, 'duration'].median():.1f} min vs other weekdays "
          f"{train_trips.loc[mask & ~is_holiday, 'duration'].median():.1f} min")
per_day = train_trips.groupby(pickup_tr.dt.date).size().rename("trips").to_frame()
per_day["holiday"] = per_day.index.isin(holiday_dates)
per_day["weekday"] = pd.to_datetime(per_day.index).day_of_week < 5
print(f"Trips per day: weekday holidays {per_day.loc[per_day.holiday, 'trips'].mean():,.0f} vs "
      f"other weekdays {per_day.loc[per_day.weekday & ~per_day.holiday, 'trips'].median():,.0f} (median)")

daily = jan.groupby(jan["tpep_pickup_datetime"].dt.day).size()
print(f"Clean trips per day: {daily.min():,} (Jan {daily.idxmin()}) to {daily.max():,} (Jan {daily.idxmax()})")
del train_trips, pickup_tr, is_holiday, is_weekday, morning

## What the exploration decides

Each number comes from a cell output above (cell named in brackets).

| Finding | Decision |
|---|---|
| 97.9% of raw trips last 1–60 min [1.1 share cell]. The rest are mostly broken: 124 negative durations, 39,987 trips under 1 min (66% with zero distance), and 3–24 h trips at a median 0.1 mph [1.1 duration-band table]. | Keep only trips of **1–60 minutes** (`clean_trips`). |
| Trips of 1–3 h are mostly real (median 17.4 miles at 14.7 mph) but only 0.9% of records [1.1 duration-band table]. | Also cut at 60 min: the model and the API cover trips of **up to 60 minutes**. |
| Fare and total amount follow the duration closely (Spearman 0.86 and 0.82), but they are recorded when the trip ends [Columns known only after the trip]. | **No fare, tip or total features.** |
| `trip_distance` also follows the duration (Spearman 0.81) and is recorded at drop-off [Columns known only after the trip]. It is also unreliable: 2.6% zero, and 162 trips over 100 miles at a median 160,212 mph [1.1 distance cells]. | **No `trip_distance`.** Distance comes from zone centers: `straight_dist` plus the pickup and drop-off zone coordinates. |
| Median duration goes from 10.4 min (6 am) to 12.2 min (2 pm), and from 10.8 min (Sunday) to 12.1 min (Thursday) [Time patterns]. | **`hour` and `day_of_week` features.** |
| On weekday holidays (Jan 1, Jan 20) there is no morning rush: 8–10 am trips take 9.4 min vs 12.0 min on other weekdays, with 88,445 vs 110,440 trips per day [Time patterns]. | **`is_holiday` feature** (US federal holidays). |
| Volume changes from day to day (77,924 trips on Jan 5 to 135,768 on Jan 16) and duration depends on the day and hour [Time patterns], so nearby trips share traffic conditions. | **Time-based split**, no shuffling: train Jan 1–20 (61.3%), validation Jan 21–25 (18.7%), test Jan 26–31 (20.0%) [3. split table]. |

Next step: `uv run nyc-taxi all` trains, compares and registers the models.

## 5. Features (shared with the API: `src/features.py`)

| Feature | Source |
|---|---|
| `PULocationID`, `DOLocationID` | request |
| `hour`, `day_of_week`, `is_holiday` | pickup time (US federal holidays) |
| `PU_lat`, `PU_lon`, `DO_lat`, `DO_lon` | center of the TLC zone shape |
| `straight_dist` | miles between the two zone centers |

**Leakage check.** Every feature is known at pickup time. `trip_distance`, fares and drop-off time are
not used. The zone table (`zone_xy`) is fixed geography from the TLC shapefile, not learned from trips,
and there are **no target statistics** (such as zone-pair mean durations) in the features. The only
statistics learned from trips are the naive baselines below, and they use training trips only. Zones
without a shape (264 "Unknown", 265 "Outside NYC") get `-1`.

In [ ]:
zone_xy = build_zone_xy(ROOT / "data")  # downloads the zone shapefile on first use
X = build_features(jan, zone_xy)
y = jan["duration"]
assert list(X.columns) == FEATURES

# Feature availability: share of trips with an unknown zone (features set to -1), per split
print((X["straight_dist"] == -1).groupby(split).mean().map("{:.2%}".format).to_string())
X.head()

## 6. Model results (read from MLflow)

Nothing is trained or computed below: each cell reads the latest finished run of its name, and says so
if a run is missing. Every model learned from the training days only (the Random Forests from the same
training sample of those days) and is compared on **all validation trips** (Jan 21–25). The test days
appear only in section 7, scored once by `nyc-taxi select`.

In [ ]:
MISSING = bl.MISSING_RUN_HINT


def load_run(name):
    """Latest finished MLflow run with this name, or None with a short message."""
    try:
        run = tracking.latest_run(name)
    except LookupError:
        print(f"No finished '{name}' run in MLflow. {MISSING}")
        return None
    print(f"{name}: run {run.info.run_id} ({pd.Timestamp(run.info.start_time, unit='ms'):%Y-%m-%d %H:%M} UTC)")
    return run


baseline_runs, missing = bl.read_baseline_runs()  # {label: run}, [missing run names]
if missing:
    print(bl.missing_runs_message(missing))
baselines = bl.baseline_metrics_table(baseline_runs) if baseline_runs else pd.DataFrame()
tuning_run = load_run(TUNING_RUN_NAME)
select_run = load_run(SELECT_RUN_NAME)
val_results = {}  # model -> {"rmse", "mae"} on validation (Jan 21-25)

### 6.1 Baselines (validation)

From runs `baseline-overall-median`, `baseline-zone-pair-median` and `rf-baseline` (`nyc-taxi train`):

- **Overall median:** the median duration of the training trips, for every trip.
- **Zone-pair median:** the median duration per pickup/drop-off zone pair of the training trips; unseen
  pairs get the overall median.
- **RF baseline:** `BASELINE_PARAMS` (50 trees, `max_depth=20`, `min_samples_leaf=5`) fitted on the
  training sample; `model_mb` is its serialized size.

In [ ]:
if baseline_runs:
    for label in baselines.index:
        val_results[label] = {"rmse": baselines.at[label, "val_rmse"] if "val_rmse" in baselines else np.nan,
                              "mae": baselines.at[label, "val_mae"] if "val_mae" in baselines else np.nan}
    if "RF baseline" in baseline_runs:
        print("RF baseline params:", {k: baseline_runs["RF baseline"].data.params.get(k) for k in BASELINE_PARAMS})
    display(baselines.reindex(columns=["val_rmse", "val_mae", "model_mb"]).round(4))
else:
    print(MISSING)

### 6.2 Optuna search

From run `optuna-tuning` (`nyc-taxi tune`): 20 TPE trials, each fitted on the training sample and scored
on the validation days; one child run per trial with its params, `val_rmse`, `val_mae` and `model_mb`.
**The test days are never used.**

In [ ]:
if tuning_run is not None:
    trials = tracking.child_runs(tuning_run.info.run_id)
    trials["trial"] = trials["tags.mlflow.runName"].str.removeprefix("trial-").astype(int)
    trials = trials.sort_values("trial").set_index("trial")

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    axes[0].plot(trials.index, trials["metrics.val_rmse"], "o", alpha=0.6, label="trial")
    axes[0].plot(trials.index, trials["metrics.val_rmse"].cummin(), color="tab:red", label="best so far")
    axes[0].set(xlabel="trial", ylabel="validation RMSE (min)", title="Optuna search")
    axes[0].legend()
    if "metrics.model_mb" in trials:
        axes[1].scatter(trials["metrics.model_mb"], trials["metrics.val_rmse"])
        axes[1].set(xscale="log", xlabel="model size (MB, log scale)", ylabel="validation RMSE (min)",
                    title="Accuracy vs size of the trials")
    plt.tight_layout()
    plt.show()

    columns = ["metrics.val_rmse", "metrics.val_mae", "metrics.model_mb",
               "params.n_estimators", "params.max_depth", "params.min_samples_leaf", "params.max_features"]
    display(trials.reindex(columns=columns).rename(columns=lambda c: c.split(".", 1)[1]).round(4))

**Best trial:** the lowest validation RMSE of the search. The model that is finally selected can be a
different trial (or the RF baseline): runs within 1% of the best RMSE count as tied and the smallest
model wins (see `model_serving_selection.ipynb`).

In [ ]:
if tuning_run is not None:
    best = trials.loc[trials["metrics.val_rmse"].idxmin()]
    print(f"Best trial: trial-{best.name}  val RMSE {best['metrics.val_rmse']:.4f}")
    val_results["Best Optuna trial"] = {"rmse": best["metrics.val_rmse"], "mae": best.get("metrics.val_mae", np.nan)}

### 6.3 Validation comparison

In [ ]:
if val_results:
    val_table = pd.DataFrame(val_results).T.rename(columns={"rmse": "val_rmse", "mae": "val_mae"})
    display(val_table.round(4))
    val_table["val_rmse"].plot.barh(figsize=(7, 3), title="Validation RMSE (Jan 21-25, minutes)")
    plt.gca().invert_yaxis()
    plt.show()
else:
    print(MISSING)

## 7. Selection and the one test score

From run `select` (`nyc-taxi select`). The rule picks the winner on validation; the winner's settings are
fitted again on the same training sample (same seed, so the same model) and scored **once** on the test
days (Jan 26–31). The baselines are scored on the same test days, so the table compares everything on
the same trips. Error segments: short (< 5 min), long (> 30 min), airport (EWR/JFK/LGA) and uncommon
zone pairs (< 10 training trips).

In [ ]:
test_table = pd.DataFrame()
if select_run is not None:
    sm, sp = select_run.data.metrics, select_run.data.params
    print(f"Winner: {sp.get('winner')} | val RMSE {sm.get('winner_val_rmse', np.nan):.4f} | "
          f"test RMSE {sm.get('test_rmse', np.nan):.4f} | {sm.get('winner_model_mb', np.nan):.1f} MB | "
          f"registered version {select_run.data.tags.get('registered_version')}")
    test_table = pd.read_csv(tracking.download_run_file(select_run.info.run_id, TEST_FILE), index_col="model")
    display(test_table[["val_rmse", "test_rmse", "test_mae"]].round(4))
    display(pd.DataFrame({
        "rmse": {s: sm.get(f"test_{s}_rmse", np.nan) for s in ev.SEGMENTS},
        "mae": {s: sm.get(f"test_{s}_mae", np.nan) for s in ev.SEGMENTS},
    }).round(3))
else:
    print(MISSING)

## 8. Results summary

In [ ]:
# Validation: every model, trained on Jan 1-20. Test: the winner and the baselines, scored once.
if val_results or not test_table.empty:
    summary = pd.DataFrame(val_results, index=["rmse", "mae"]).T.add_prefix("val_")
    if not test_table.empty:
        summary = summary.join(test_table[["test_rmse", "test_mae"]], how="outer")
    display(summary.round(4))
else:
    print(MISSING)

## 9. Conclusion

- **Selected model:** the winner of the rule in `nyc-taxi select` (section 7). Its test RMSE/MAE are the
  only test numbers, computed once, next to the baselines on the same days.
- **Against naive baselines:** compare the winner's test RMSE with the zone-pair median (section 7).
- **No leakage:** features are known at pickup; every statistic and every model comes from the training
  days only; validation chose the model; the test days were scored once.
- **Limits:** 1–60 minute trips only, one month of data (no seasons or weather), and zone-level location
  only, so trips within a zone pair vary a lot. Errors grow for long and airport trips (section 7).

*Update the numbers in this conclusion after running the notebook if you quote them elsewhere.*